# Notebook 07: Feed-Forward Networks and Activations

## Why This Matters

In a transformer, the FFN layers collectively hold roughly two-thirds of all parameters — far more than the attention mechanism. The activation function choice inside the FFN significantly impacts model quality, and modern LLMs all use SwiGLU or a gated variant rather than the original ReLU. Understanding why requires understanding the "FFN as memory" hypothesis (Geva et al. 2021), which explains what these massive FFN layers are actually doing and why making them larger improves factual knowledge. This is also where Mixture of Experts (MoE) models like Mixtral make their intervention — replacing dense FFN layers with sparse routed experts for much more parameter-efficient scaling.

## Background

The original transformer FFN was:
$$\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 x + b_1) + b_2$$

Modern LLMs use gated variants like SwiGLU:
$$\text{FFN}_{\text{SwiGLU}}(x) = (W_1 x \odot \text{SiLU}(W_3 x)) \cdot W_2$$

The intuition: gating introduces a learned "filter" that controls which activations pass through, giving the model richer expressiveness per parameter.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. The FFN as Key-Value Memory

Geva et al. (2021) showed that FFN layers in transformers behave like key-value memories:

$$\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 x)$$

- **Keys** = rows of $W_1$ (shape $d_{\text{model}} \times d_{\text{ff}}$): each neuron in the hidden layer acts as a pattern detector
- **Values** = rows of $W_2$ (shape $d_{\text{ff}} \times d_{\text{model}}$): each neuron's corresponding output contribution

When $x$ has high similarity with key $w_1^{(i)}$, neuron $i$ activates, and the corresponding value $w_2^{(i)}$ is added to the output. This is exactly the mechanism of a soft key-value retrieval.

**Why this matters:** Factual knowledge (e.g., "Paris is the capital of France") is stored in FFN weights. Larger FFN layers → more "memory slots" → more factual knowledge. This is why scaling FFN size (not just attention size) improves factual accuracy. It also explains why model editing techniques like ROME target FFN weights to change specific facts.

**Sparsity observation:** After training, ReLU activations are extremely sparse — only ~5% of neurons activate for any given input. SwiGLU reduces but doesn't eliminate this sparsity.

In [ ]:
# Visualize the key-value memory interpretation
d_model, d_ff = 64, 256
W1 = nn.Linear(d_model, d_ff, bias=False)
W2 = nn.Linear(d_ff, d_model, bias=False)

# Simulate a diverse batch
x_batch = torch.randn(32, 20, d_model)

# Track activation patterns
with torch.no_grad():
    hidden = F.relu(W1(x_batch))  # (32, 20, d_ff)
    # Which neurons activate most?
    activation_rates = (hidden > 0).float().mean(dim=(0, 1))  # (d_ff,)
    activation_magnitudes = hidden.mean(dim=(0, 1))           # (d_ff,)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(d_ff), activation_rates.numpy(), alpha=0.7)
axes[0].set_xlabel("Neuron index")
axes[0].set_ylabel("Activation rate")
axes[0].set_title(f"ReLU neuron activation rates (mean={activation_rates.mean():.2f})")
axes[0].axhline(y=activation_rates.mean().item(), color='red', linestyle='--',
                label=f"Mean={activation_rates.mean():.2f}")
axes[0].legend()

axes[1].scatter(range(d_ff), activation_magnitudes.numpy(),
                c=activation_rates.numpy(), cmap='hot', alpha=0.7, s=10)
axes[1].set_xlabel("Neuron index")
axes[1].set_ylabel("Mean activation magnitude")
axes[1].set_title("Neuron specialization (darker = more active)")

plt.suptitle("FFN as Sparse Key-Value Memory: Neuron Activation Statistics", fontsize=12)
plt.tight_layout()
plt.savefig("ffn_memory.png", dpi=120, bbox_inches='tight')
plt.show()

## 2. Activation Functions: ReLU → GELU → SwiGLU

**ReLU** (Rectified Linear Unit): $\text{ReLU}(x) = \max(0, x)$
- Simple, fast, creates hard sparsity
- "Dying ReLU" problem: neurons with negative input never activate, gradient = 0

**GELU** (Gaussian Error Linear Unit): $\text{GELU}(x) = x \cdot \Phi(x)$ where $\Phi$ is the Gaussian CDF
- Approximated as $0.5x(1 + \tanh(\sqrt{2/\pi}(x + 0.044715x^3)))$
- Smooth, differentiable everywhere; small gradient even for negative $x$
- Used in BERT, GPT-2, original ViT

**SiLU** (Sigmoid Linear Unit, "Swish"): $\text{SiLU}(x) = x \cdot \sigma(x)$
- Similar to GELU but slightly different shape; computationally cheaper

**SwiGLU** (Gated variant using SiLU, Shazeer 2020):
$$\text{SwiGLU}(x, W, V) = \text{SiLU}(xW) \odot (xV)$$

The gate $\text{SiLU}(xW)$ learns to selectively amplify or suppress the linear transform $xV$. This adds expressiveness without adding more layers — the gating mechanism creates a learned data-dependent nonlinearity.

**Why SwiGLU wins:** The PaLM paper (Chowdhery et al. 2022) showed SwiGLU consistently outperforms GELU and ReLU across model sizes. LLaMA, Mistral, Gemma, and Qwen all use SwiGLU. The downside: it requires 3 weight matrices instead of 2, so the intermediate dimension is typically set to $\frac{2}{3} \times 4d = \frac{8}{3}d$ to keep parameter count comparable.

In [ ]:
def relu(x): return F.relu(x)
def gelu(x): return F.gelu(x)
def silu(x): return F.silu(x)  # = Swish

# Visualize activation functions
x = torch.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, fn, color in [('ReLU', relu, 'blue'), ('GELU', gelu, 'green'), ('SiLU', silu, 'red')]:
    y = fn(x).numpy()
    axes[0].plot(x.numpy(), y, label=name, color=color, linewidth=2)

axes[0].set_title("Activation Functions")
axes[0].set_xlabel("x")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)

# Gradients
x_grad = x.clone().requires_grad_(True)
for name, fn, color in [('ReLU', relu, 'blue'), ('GELU', gelu, 'green'), ('SiLU', silu, 'red')]:
    x_grad = x.clone().requires_grad_(True)
    y = fn(x_grad)
    y.sum().backward()
    axes[1].plot(x.numpy(), x_grad.grad.numpy(), label=f'd/dx {name}', color=color, linewidth=2)

axes[1].set_title("Activation Gradients")
axes[1].set_xlabel("x")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylim(-0.2, 1.2)

plt.suptitle("ReLU vs GELU vs SiLU: Smooth nonlinearities improve gradient flow", fontsize=12)
plt.tight_layout()
plt.savefig("activations.png", dpi=120, bbox_inches='tight')
plt.show()

## 3. Implementing Dense FFN and SwiGLU

In [ ]:
class FFNWithGELU(nn.Module):
    """Standard transformer FFN with GELU activation."""
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.linear1 = nn.Linear(d_model, d_ff, bias=False)
        self.linear2 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

class SwiGLUFFN(nn.Module):
    """
    SwiGLU FFN: used in LLaMA, Mistral, Gemma, Qwen.
    Uses 3 weight matrices; d_ff typically set to 8/3 * d_model
    to match parameter count of standard 4x FFN.
    """
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        # 8/3 * d_model to match params of 4x standard FFN with 2 matrices
        if d_ff is None:
            d_ff = int(8/3 * d_model)
            # Round to multiple of 64 for hardware efficiency
            d_ff = 64 * math.ceil(d_ff / 64)
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)   # SiLU gate
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)   # Linear
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)   # Output projection
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU: SiLU(gate) * up_proj, then project down
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(self.dropout(gate * up))

# Compare parameter counts
d_model = 512
ffn_gelu = FFNWithGELU(d_model)
ffn_swiglu = SwiGLUFFN(d_model)

gelu_params = sum(p.numel() for p in ffn_gelu.parameters())
swiglu_params = sum(p.numel() for p in ffn_swiglu.parameters())
d_ff_gelu = 4 * d_model
d_ff_swiglu = 64 * math.ceil(int(8/3*d_model)/64)

print(f"GELU FFN:   d_ff={d_ff_gelu}, params={gelu_params:,}")
print(f"SwiGLU FFN: d_ff={d_ff_swiglu}, params={swiglu_params:,}")
print(f"Param ratio (SwiGLU/GELU): {swiglu_params/gelu_params:.3f}")

# Forward pass equivalence test
x = torch.randn(4, 16, d_model)
out_gelu = ffn_gelu(x)
out_swiglu = ffn_swiglu(x)
print(f"\nGELU FFN output shape:   {out_gelu.shape}")
print(f"SwiGLU FFN output shape: {out_swiglu.shape}")

# Measure sparsity in hidden activations
with torch.no_grad():
    hidden_gelu   = F.gelu(ffn_gelu.linear1(x))
    gate_swiglu   = F.silu(ffn_swiglu.gate_proj(x))
    up_swiglu     = ffn_swiglu.up_proj(x)
    hidden_swiglu = gate_swiglu * up_swiglu
    near_zero_gelu   = (hidden_gelu.abs() < 0.01).float().mean()
    near_zero_swiglu = (hidden_swiglu.abs() < 0.01).float().mean()
    print(f"\nNear-zero hidden GELU:   {near_zero_gelu.item():.3f}")
    print(f"Near-zero hidden SwiGLU: {near_zero_swiglu.item():.3f}")

## 4. Parameter Counts and FLOPs for Attention vs FFN

Understanding the breakdown of parameters and computation is essential for architecture decisions and cost estimation.

In [ ]:
def transformer_param_count(d_model, n_heads, n_layers, vocab_size,
                             ffn_type='gelu', use_biases=False):
    """Compute parameter breakdown for a transformer model."""
    d_ff_gelu   = 4 * d_model
    d_ff_swiglu = 64 * math.ceil(int(8/3 * d_model) / 64)
    d_ff = d_ff_swiglu if ffn_type == 'swiglu' else d_ff_gelu
    n_ffn_matrices = 3 if ffn_type == 'swiglu' else 2

    # Embedding
    embedding_params = vocab_size * d_model

    # Per layer
    mha_params = 4 * d_model * d_model   # Q, K, V, O (no bias)
    ffn_params  = n_ffn_matrices * d_model * d_ff
    norm_params = 2 * d_model  # 2 LayerNorms per layer (gamma only for RMSNorm = d_model each)

    per_layer = mha_params + ffn_params + norm_params
    total_layers = n_layers * per_layer

    total = embedding_params + total_layers
    # Note: weight tying means LM head shares with embedding (no extra params)

    return {
        "embedding": embedding_params,
        "per_layer_mha": mha_params,
        "per_layer_ffn": ffn_params,
        "per_layer_total": per_layer,
        "total_layers": total_layers,
        "total": total,
        "ffn_fraction": n_layers * ffn_params / total_layers,
        "d_ff": d_ff,
    }

# Print breakdown for various model sizes
print("Parameter breakdown (with weight tying, no biases):")
print("="*70)
configs = [
    ("GPT-2 Small",   768,  12, 12, 50257, 'gelu'),
    ("GPT-2 Large",  1280,  20, 36, 50257, 'gelu'),
    ("LLaMA-7B",     4096,  32, 32, 32000, 'swiglu'),
    ("LLaMA-70B",    8192,  64, 80, 32000, 'swiglu'),
]
for name, d_model, n_heads, n_layers, vocab, ffn_type in configs:
    stats = transformer_param_count(d_model, n_heads, n_layers, vocab, ffn_type)
    print(f"\n{name} (d={d_model}, L={n_layers}, {ffn_type}):")
    print(f"  Embedding:  {stats['embedding']/1e6:.1f}M")
    print(f"  Layers:     {stats['total_layers']/1e6:.1f}M")
    print(f"  Total:      {stats['total']/1e9:.2f}B")
    print(f"  FFN frac:   {stats['ffn_fraction']:.1%} of transformer layers")
    print(f"  d_ff:       {stats['d_ff']}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| FFN as memory | Rows of $W_1$ = keys; rows of $W_2$ = values; neurons store factual patterns |
| FFN parameter share | ~67% of transformer layer parameters live in FFN (not attention) |
| ReLU | Simple, fast, sparse; dying neuron problem; not used in modern LLMs |
| GELU | Smooth, differentiable everywhere; used in BERT, GPT-2; $x \cdot \Phi(x)$ |
| SiLU | $x \cdot \sigma(x)$; similar to GELU; slightly more efficient |
| SwiGLU | Gated FFN: $\text{SiLU}(W_1 x) \odot W_3 x$; 3 weight matrices; best empirical quality |
| SwiGLU d_ff | $\frac{8}{3} d_{\text{model}}$ (not $4 \times$) to keep params comparable to GELU FFN |
| MoE | Replace dense FFN with sparse routed experts; Mixtral uses 8 experts, activates 2 |
| Modern standard | LLaMA, Mistral, Gemma, Qwen all use SwiGLU + RMSNorm |